In [3]:
######################################################################################
# Authored by Andrew Wan Ju Kang, Ph.D.
# soarhigh0714@gmail.com
# Do not re-distribute.
######################################################################################

In [4]:
# ─── llama-server 연결 설정 ─────────────────────────────────────────────
# 본 노트북은 외부 docker 컨테이너로 띄운 llama-server 에 붙는 클라이언트입니다.
# 노트북 실행 전, 별도 터미널에서 먼저 서버를 띄워두세요:
#
#     bash start_llama_server.sh
#
# 모델 다운로드 (-hf), GPU 로딩, mmproj 자동 결합은 모두 서버 쪽에서 처리됩니다.
# 클라이언트(노트북)는 host:port 만 알면 됩니다.

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8181


In [5]:
from langchain.tools import tool
from pydantic import BaseModel, Field

# 1일차 실습에서
# 상품평 주어지고, summary 필드에 요약문, sentiment 필드에 감정


# response format -> ToolStrategy
# pydantic 라이브러리 활용
# pydantic이 형식을 강제.
class WeatherInput(BaseModel):
    """날씨 조회를 위한 입력 스키마"""
    location: str = Field(description="날씨를 조회할 도시 이름 (예: '서울', 'Tokyo')")
# tool 선택할 때 LLM이 고려하는 것:
# tool 이름 + (만약 존재한다면) 하위 필드 이름 + Field의 묘사 (description)
# 프롬프트
# tool docstring


@tool("get_weather", args_schema=WeatherInput)
def get_weather(location: str) -> str:
    """주어진 도시의 현재 날씨 정보를 조회합니다.
    도시 이름을 입력하면 온도, 상태, 습도를 반환합니다."""
    weather_data = {
        "서울": {"temp": "22°C", "condition": "맑음", "humidity": "45%"},
        "부산": {"temp": "25°C", "condition": "흐림", "humidity": "70%"},
        "도쿄": {"temp": "28°C", "condition": "비", "humidity": "80%"},
        "Tokyo": {"temp": "28°C", "condition": "Rainy", "humidity": "80%"},
        "뉴욕": {"temp": "18°C", "condition": "구름 조금", "humidity": "55%"},
        "New York": {"temp": "18°C", "condition": "Partly Cloudy", "humidity": "55%"},
    }
    if location in weather_data:
        d = weather_data[location]
        return f"{location} 날씨: 온도 {d['temp']}, 상태 {d['condition']}, 습도 {d['humidity']}"
    return f"{location}의 날씨 정보를 찾을 수 없습니다. 사용 가능 도시: {', '.join(weather_data.keys())}"


@tool("get_user_location")
def get_user_location() -> str:
    """현재 사용자의 위치를 조회합니다.
    별도 입력 없이 호출하면 사용자의 현재 도시를 반환합니다."""
    return "서울"


class CalculatorInput(BaseModel):
    """계산기 입력 스키마"""
    expression: str = Field(description="계산할 수학 표현식 (예: '2 + 3 * 4')")


@tool("calculator", args_schema=CalculatorInput)
def calculator(expression: str) -> str:
    """수학 표현식을 계산합니다. 사칙연산, 제곱, 괄호 등을 지원합니다."""
    try:
        allowed_chars = set("0123456789+-*/.() ")    # 집합 (duplicate 없이)
        if not all(c in allowed_chars for c in expression):
            return "오류: 허용되지 않는 문자가 포함되어 있습니다."
        result = eval(expression)                    # 문자열 내용의 값을 연산 ("1+1" -> 2)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


# 사용할 도구 리스트
tools = [get_weather, get_user_location, calculator]

print(f"🔧 등록된 도구 ({len(tools)}개): {[t.name for t in tools]}")

🔧 등록된 도구 (3개): ['get_weather', 'get_user_location', 'calculator']


/opt/conda/lib/python3.10/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# ─── LLM 연결: llama-server → OpenAI 호환 API ───

# ChatOpenAI는 OpenAI 모델 지원 (당연)
# 이외의 모델이더라도 OpenAI API "형식"을 갖추면 공통 지원
llm = ChatOpenAI(
    model="gemma",                                       # 모델명 alias와 같게.
    base_url=f"http://{SERVER_HOST}:{SERVER_PORT}/v1",          # 로컬 llama-server
    api_key="not-needed",                                       # 로컬 서버이므로 불필요
    temperature=0.1,
    max_tokens=1024,  # max_tokens는 생성 (출력) 길이 제한
)


# ─── 시스템 프롬프트 ───
SYSTEM_PROMPT = """You are a helpful AI assistant with access to tools.

When the user asks about weather, use the available tools to get the information.
If you need the user's location first, call get_user_location, then use that result
to call get_weather.

When the user asks a math question, use the calculator tool.

Always respond in Korean. Think step by step."""

# ─── ReAct 에이전트 생성 ───
# create_agent 시그니처 (공식 API Reference):
#   create_agent(
#       model: str | BaseChatModel,
#       tools: Sequence[BaseTool | Callable | dict] | None,
#       *, system_prompt, middleware, response_format, ...
#   ) -> CompiledStateGraph
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ LangChain v1.0 ReAct 에이전트 생성 완료!")

✅ LangChain v1.0 ReAct 에이전트 생성 완료!


In [8]:
import base64
import mimetypes


def run_agent(query: str, image_url: str = None, verbose: bool = True):
    """에이전트를 invoke 방식으로 실행합니다.

    Args:
        query: 질문 텍스트
        image_url: 이미지 URL 또는 로컬 파일 경로 (선택, 멀티모달)
        verbose: 중간 단계 출력 여부
    """
    print(f"\n{'='*60}")
    print(f"📝 질문: {query}")
    if image_url:
        print(f"🖼️ 이미지: {image_url}")
    print(f"{'='*60}")

    # ─── 메시지 구성 ───
    if image_url:
        if not image_url.startswith("http"):
            mime = mimetypes.guess_type(image_url)[0] or "image/png"
            with open(image_url, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            url = f"data:{mime};base64,{b64}"
        else:
            url = image_url

        user_message = {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": query},
            ],
        }
    else:
        user_message = {"role": "user", "content": query}

    # ─── 에이전트 실행 (invoke) ───
    result = agent.invoke({"messages": [user_message]})

    # ─── 결과 출력 ───
    if verbose:
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🔧 Tool Call: {tc['name']}({tc['args']})")
            elif hasattr(msg, "type") and msg.type == "tool":
                print(f"  📦 Tool Result [{msg.name}]: {msg.content}")

    final_response = result["messages"][-1].content
    print(f"\n🎯 최종 응답:\n{final_response}")
    return final_response

print("✅ 실행 함수 정의 완료!")

✅ 실행 함수 정의 완료!


In [9]:
# 텍스트만 (기존과 동일)
run_agent("서울 날씨 어때?")

# 웹 이미지 URL
run_agent(
    "이 이미지에 무엇이 보이는지 설명해줘",
    image_url="/home/user/MultiAgents/Day_2/Ex/image_data/cat.jpg"
)


📝 질문: 서울 날씨 어때?
  🔧 Tool Call: get_weather({'location': '서울'})
  📦 Tool Result [get_weather]: 서울 날씨: 온도 22°C, 상태 맑음, 습도 45%

🎯 최종 응답:
서울 날씨는 온도 22°C, 상태 맑음, 습도 45%입니다.

📝 질문: 이 이미지에 무엇이 보이는지 설명해줘
🖼️ 이미지: /home/user/MultiAgents/Day_2/Ex/image_data/cat.jpg

🎯 최종 응답:
이미지에는 실내에서 편안하게 쉬고 있는 고양이 한 마리가 보입니다.

**자세한 설명:**

*   **주제:** 고양이 한 마리
*   **고양이의 모습:** 고양이는 털이 검은색과 회색이 섞인 얼룩무늬(태비 무늬)를 가지고 있으며, 편안하게 엎드려 있습니다. 앞발을 앞으로 쭉 뻗고 눈을 감거나 편안하게 쉬고 있는 듯한 모습입니다.
*   **배경:**
    *   고양이는 나무로 된 표면(테이블이나 널빤지) 위에 놓여 있습니다.
    *   고양이 아래에는 부드러운 질감의 회색 담요나 천이 깔려 있어 따뜻하고 아늑한 분위기를 연출합니다.
    *   왼쪽 배경으로는 큰 창문이 보이며, 창밖으로는 푸른 녹음과 나무들이 보여 자연광이 들어오고 있습니다.
    *   창문 프레임과 실내 인테리어 요소들이 보입니다.

전반적으로 이 이미지는 **평화롭고 따뜻하며, 고양이가 편안하게 휴식을 취하는 모습**을 담고 있습니다.


'이미지에는 실내에서 편안하게 쉬고 있는 고양이 한 마리가 보입니다.\n\n**자세한 설명:**\n\n*   **주제:** 고양이 한 마리\n*   **고양이의 모습:** 고양이는 털이 검은색과 회색이 섞인 얼룩무늬(태비 무늬)를 가지고 있으며, 편안하게 엎드려 있습니다. 앞발을 앞으로 쭉 뻗고 눈을 감거나 편안하게 쉬고 있는 듯한 모습입니다.\n*   **배경:**\n    *   고양이는 나무로 된 표면(테이블이나 널빤지) 위에 놓여 있습니다.\n    *   고양이 아래에는 부드러운 질감의 회색 담요나 천이 깔려 있어 따뜻하고 아늑한 분위기를 연출합니다.\n    *   왼쪽 배경으로는 큰 창문이 보이며, 창밖으로는 푸른 녹음과 나무들이 보여 자연광이 들어오고 있습니다.\n    *   창문 프레임과 실내 인테리어 요소들이 보입니다.\n\n전반적으로 이 이미지는 **평화롭고 따뜻하며, 고양이가 편안하게 휴식을 취하는 모습**을 담고 있습니다.'